In [0]:
%pip install openai
%pip install langchain-text-splitters

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 580.5/580.5 kB 22.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 940.8/940.8 kB 41.4 MB/s  0:00:00
  Attempting uninstall: jsonpointer
    Found existing installation: jsonpointer 3.1.1
    Not uninstalling jsonpointer at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-4f90126a-7484-4a83-9ef8-cfdba1e43f00
    Can't uninstall 'jsonpointer'. No files were found to uninstall.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [langchain-text-splitters]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

In [0]:
from pyspark.sql.functions import *    
from pyspark.sql.types import *
from datetime import * 
from openai import OpenAI
import os
import json

In [0]:
df_customer = spark.read\
    .option("header" , True)\
    .option("inferschema" , True)\
    .csv("/Volumes/retail_data/bronze/raw_data_volume/gen_ai_datasets/customer_data_genai.csv")

df_employee = spark.read\
    .option("header" , True)\
    .option("inferschema" , True)\
    .csv("/Volumes/retail_data/bronze/raw_data_volume/gen_ai_datasets/employee_data_genai_final.csv")

df_payment  =  spark.read\
    .option("header" , True)\
    .option("inferschema" , True)\
    .csv("/Volumes/retail_data/bronze/raw_data_volume/gen_ai_datasets/payment_data_genai.csv")

Error in callback <bound method UserNamespaceCommandHook.post_run_cell of <dbruntime.DatasetInfo.UserNamespaceCommandHook object at 0xffdfa58829f0>> (for post_run_cell), with arguments args (<ExecutionResult object at ffdf93bcf530, execution_count=4 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at ffdf93bcd010, raw_cell="df_customer = spark.read\
    .option("header" , T.." transformed_cell="df_customer = spark.read\
    .option("header" , T.." store_history=True silent=False shell_futures=True cell_id=8406422638858143> result=None>,),kwargs {}:


In [0]:
df_customer_updated = df_customer.withColumn("salary" , expr("try_cast(salary as int)"))\
                         .withColumn("age" , expr("try_cast(age as int)"))\
                         .withColumn("created_at" , expr("try_cast(created_at as timestamp)"))\
                         .withColumn("email" , expr("try_cast(email as string)"))\
                         .withColumn("Time_stamp" , current_timestamp())

df_employee_updated = df_employee.withColumn("salary" , expr("try_cast(salary as int)"))\
                         .withColumn("manager_id" , expr("try_cast(manager_id as int)"))\
                         .withColumn("joining_date" , expr("try_cast(joining_date as date)"))\
                         .withColumn("bonus" , expr("try_cast(bonus as int)"))\
                         .withColumn("Time_stamp" , current_timestamp())

df_payment_updated  = df_payment.withColumn("amount" , expr("try_cast(amount as int)"))\
                       .withColumn("payment_date" , expr("try_cast(payment_date as timestamp)"))\
                       .withColumn("Time_stamp" , current_timestamp())

In [0]:
a1 = df_customer_updated.count()
a2 = df_customer_updated.dropna(subset = ["customer_id" , "email" , "name"]).count()

b1 = df_employee_updated.count()
b2 = df_employee_updated.dropna(subset = ["emp_id" , "emp_name" , "department"]).count()

c1 = df_payment_updated.count()
c2 = df_payment_updated.dropna(subset = ["payment_id" , "transaction_reference" , "payment_status" , "customer_id"]).count()

print(a1 , f" , Bad Records of Customer data: {a1-a2}" , f" , After Bad records Removed {a2}")
print(b1 , f" , Bad Records of Employee data  : {b1-b2}" , f" , After Bad records Removed {b2}")
print(c1 , f" , Bad Records of Employee Data : {c1-c2}" , f" , After Bad records Removed {c2}")


1520  , Bad Records of Customer data: 42  , After Bad records Removed 1478
1500  , Bad Records of Employee data  : 4  , After Bad records Removed 1496
2000  , Bad Records of Employee Data : 41  , After Bad records Removed 1959


In [0]:

############### Quarantine Bad Records ####################################

df_customer_quarantine = df_customer_updated.filter(col("customer_id").isNull() \
                                                 | col("email").isNull() \
                                                 | col("name").isNull())

df_employee_quarantine = df_employee_updated.filter(col("emp_id").isNull() \
                                                  | col("emp_name").isNull() \
                                                  | col("department").isNull())

df_payment_quarantine = df_payment_updated.filter(col("payment_id").isNull() \
                                                | col("transaction_reference").isNull()\
                                                | col("payment_status").isNull()
                                                | col("customer_id").isNull())



####################### Quarantine data write in bronze table #############################

df_customer_quarantine.write\
    .mode("overwrite")\
    .option("mergeSchema" , True)\
    .format("delta")\
    .saveAsTable("bronze.quarantine.customer_quarantine")

df_employee_quarantine.write\
    .mode("overwrite")\
    .option("mergeSchema" , True)\
    .format("delta")\
    .saveAsTable("bronze.quarantine.employee_quarantine")

df_payment_quarantine.write\
    .mode("overwrite")\
    .option("mergeSchema" , True)\
    .format("delta")\
    .saveAsTable("bronze.quarantine.payment_quarantine")



################## Clean data write in silver table #############################33

df_customer_cleaned = df_customer_updated.dropna(subset = ["customer_id" , "email" , "name"])
df_employee_cleaned = df_employee_updated.dropna(subset = ["emp_id" , "emp_name" , "department"])
df_payment_cleaned = df_payment_updated.dropna(subset = ["payment_id" , "transaction_reference" , "payment_status" , "customer_id"])


df_customer_cleaned.write\
    .mode("overwrite")\
    .option("mergeSchema" , True)\
    .format("delta")\
    .saveAsTable("silver.cleaned.customer_cleaned")

df_employee_cleaned.write\
    .mode("overwrite")\
    .option("mergeSchema" , True)\
    .format("delta")\
    .saveAsTable("silver.cleaned.employee_cleaned")

df_payment_cleaned.write\
    .mode("overwrite")\
    .option("mergeSchema" , True)\
    .format("delta")\
    .saveAsTable("silver.cleaned.payment_cleaned")


In [0]:
df_customer_cleaned = spark.table(f"silver.cleaned.customer_cleaned")

df_customer_cleaned = df_customer_cleaned.withColumn("Loan_Account_No",
     when(
        regexp_extract(col("customer_message"), r'((?:BLU|PLU|DIG|LAP|SCF)\d+)', 1) != "",
        regexp_extract(col("customer_message"), r'((?:BLU|PLU|DIG|LAP|SCF)\d+)', 1)
    ).otherwise(None)
).drop("loan_account_number_extracted" , "feedback_sentiment")

In [0]:
df_customer_cleaned = df_customer_cleaned.withColumn("feedback_sentiment" , lit(None).cast("string"))

### Gen AI phase start


In [0]:
dbutils.secrets.list("genai-secrets")

[SecretMetadata(key='OPENAI_API_KEY')]

In [0]:
api_key = dbutils.secrets.get(
    scope = "genai-secrets" , 
    key = "OPENAI_API_KEY"
)

client  = OpenAI(api_key = api_key)


In [0]:
distint_feedback = [f.feedback_text for f in df_customer_cleaned.select("feedback_text").select("feedback_text").distinct().collect()]

print(len(distint_feedback))

15


In [0]:
feedback =  [f.feedback_text for f in df_customer_cleaned.select("feedback_text").select("feedback_text").distinct().collect()]

response = client.responses.create(
    model = "gpt-5" , 
    input =  f"""
    You are a customer feedback sentiment analyser.

    Classify each feedback into exactly one category : Positive , Negative , or Neutral

    Return the result as Json Array . 
    Do no skip any feedback .

    Requiored Json Format :
    [{{
        "feedback_text" : "Original Feedback" ,
        "sentiment" : "Positive"
    }}]

    Feedbacks : 
    {json.dumps(feedback)}
    """
)
print(response.output_text)

[
  {
    "feedback_text": "Excellent service, transparent charges, and timely updates.",
    "sentiment": "Positive"
  },
  {
    "feedback_text": "The payment failed twice and customer care did not provide a solution.",
    "sentiment": "Negative"
  },
  {
    "feedback_text": "I have called several times, but nobody has helped me. This experience is frustrating.",
    "sentiment": "Negative"
  },
  {
    "feedback_text": "I am not happy with the repeated calls and poor follow-up from the team.",
    "sentiment": "Negative"
  },
  {
    "feedback_text": "The loan process was smooth and the staff explained every step clearly.",
    "sentiment": "Positive"
  },
  {
    "feedback_text": "My issue was resolved, but only after multiple follow-ups.",
    "sentiment": "Neutral"
  },
  {
    "feedback_text": "I am very disappointed with the delays and lack of communication.",
    "sentiment": "Negative"
  },
  {
    "feedback_text": "The service was okay. The process took some time, but even

In [0]:
sentiment  =  json.loads(response.output_text)

for s in sentiment:
    df_customer_cleaned = df_customer_cleaned.withColumn(
        "feedback_sentiment" , 
        when(
            col("feedback_text") == s["feedback_text"] , 
            lit(s["sentiment"])
            ).otherwise(col("feedback_sentiment")))

In [0]:
df_customer_cleaned.display()

customer_id,name,city,state_code,salary,email,phone,age,customer_type,created_at,customer_message,feedback_text,Time_stamp,Loan_Account_No,feedback_sentiment
CUST1,Rakesh,Pune,MH,50000,rakesh1@example.com,7201798311,25,New,2025-10-20T00:00:00.000Z,Please call me regarding my loan application. I do not have the account number.,"Excellent service, transparent charges, and timely updates.",2026-11-05T06:21:46.181Z,null,Positive
CUST2,Sneha,Hyderabad,MH,40000,sneha2@example.com,6524890042,24,Premium,2025-08-30T00:00:00.000Z,I want to know the outstanding balance on BLU20260000001.,"Excellent service, transparent charges, and timely updates.",2026-11-05T06:21:46.181Z,BLU20260000001,Positive
CUST3,Rakesh,Chennai,MH,32000,rakesh3@example.com,8156676774,65,New,2024-10-13T00:00:00.000Z,I need help with account number SCF20260000002. The payment details are not reflecting correctly.,The payment failed twice and customer care did not provide a solution.,2026-11-05T06:21:46.181Z,SCF20260000002,Negative
CUST4,Kavita,Jaipur,TS,25000,kavita4@example.com,8062546254,59,Business,2024-07-18T00:00:00.000Z,There is an issue with my EMI for DIG20260000003. Please check and call me back.,"I have called several times, but nobody has helped me. This experience is frustrating.",2026-11-05T06:21:46.181Z,DIG20260000003,Negative
CUST5,Ankush,Mumbai,BR,45000,ankush5@example.com,8422996417,21,Business,2023-03-31T00:00:00.000Z,Can you send the statement for loan account LAP20260000004? I need it for my records.,I am not happy with the repeated calls and poor follow-up from the team.,2026-11-05T06:21:46.181Z,LAP20260000004,Negative
CUST6,Sneha,Chennai,DL,25000,sneha6@example.com,6786563119,61,Business,2025-02-27T00:00:00.000Z,I want to know the outstanding balance on PLU20260000005.,The loan process was smooth and the staff explained every step clearly.,2026-11-05T06:21:46.181Z,PLU20260000005,Positive
CUST7,Suresh,Hyderabad,RJ,75000,suresh7@example.com,7979816119,54,Regular,2022-08-02T00:00:00.000Z,Can you send the statement for loan account BLU20260000006? I need it for my records.,The payment failed twice and customer care did not provide a solution.,2026-11-05T06:21:46.181Z,BLU20260000006,Negative
CUST8,Sneha,Bangalore,KA,50000,sneha8@example.com,7278449359,58,Regular,2025-08-21T00:00:00.000Z,"I contacted support regarding my supply chain finance, reference SCF20260000007, but my issue is still open.",The loan process was smooth and the staff explained every step clearly.,2026-11-05T06:21:46.181Z,SCF20260000007,Positive
CUST9,Ankush,Nagpur,TN,75000,ankush9@example.com,8520014340,59,Business,2024-04-07T00:00:00.000Z,I want to know the outstanding balance on DIG20260000008.,"My issue was resolved, but only after multiple follow-ups.",2026-11-05T06:21:46.181Z,DIG20260000008,Neutral
CUST10,Mohit,Patna,UP,40000,mohit10@example.com,6316256925,34,Regular,2022-02-22T00:00:00.000Z,I want to know the outstanding balance on LAP20260000009.,"Excellent service, transparent charges, and timely updates.",2026-11-05T06:21:46.181Z,LAP20260000009,Positive
